# fitcast — Colab edition

Scrape job postings → predict if you qualify → tailor your resume.

In-browser version of [fitcast](https://github.com/aa9gj/fitcast) — **no install, no Python setup**. Just an Anthropic API key and ~5 minutes.

## What it costs

- **First run on a fresh Colab**: ~$0.50 in Anthropic API spend (10 jobs scored + 3 tailored resumes), plus ~30s of free Colab compute.
- **Subsequent runs**: same.
- See the full [cost breakdown](https://github.com/aa9gj/fitcast#what-it-costs-you) for details.

## How to use

1. Set your Anthropic API key (cells 1–2)
2. Upload your resume.md (cell 4)
3. Tweak settings (cell 6)
4. Click **Runtime → Run all**
5. Browse results, download tailored resumes

In [ ]:
# Install deps + clone the latest fitcast pipeline
# Note: while the repo is private, only the repo owner can clone it.
!pip install -q anthropic pydantic requests pyyaml sentence-transformers PyPDF2 python-docx
!git clone -q https://github.com/aa9gj/fitcast.git 2>&1 | tail -3
%cd fitcast
print("\n✓ Setup complete — fitcast cloned and ready")

## 1. Anthropic API key

Get one at [console.anthropic.com/settings/keys](https://console.anthropic.com/settings/keys) (free signup; needs ~$5 credit minimum).

**Recommended:** click the 🔑 (key) icon in Colab's left sidebar → **Add new secret** → name it `ANTHROPIC_API_KEY`. The next cell picks it up automatically and your key never appears in the notebook.

In [ ]:
import os
from getpass import getpass

api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        print("✓ Loaded API key from Colab Secrets.")
except Exception:
    pass

if not api_key:
    api_key = getpass("Paste your Anthropic API key (input is hidden): ")

os.environ["ANTHROPIC_API_KEY"] = api_key
assert api_key, "API key not set."
print(f"✓ API key set ({len(api_key)} chars).")

## 2. Your resume

Upload your `resume.md` (markdown format gives Claude the cleanest signal).

Don't have a markdown resume? Drop your PDF into [Claude.ai](https://claude.ai) and ask: *"Convert this resume to clean markdown, preserving all sections, bullets, and dates."* Save the output as `resume.md` and upload it here.

In [ ]:
from google.colab import files

print("Click 'Choose Files' and select your resume.md (or .txt).")
uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded. Please re-run this cell.")

filename = next(iter(uploaded))
content = uploaded[filename].decode("utf-8", errors="replace")
with open("resume.md", "w") as f:
    f.write(content)
print(f"\n✓ Saved {len(content)} characters to resume.md")

## 3. Configure your search

Form fields below — change values, then run the cell. Defaults are sensible for a generalist data/AI/health/biotech search.

**Tip:** want personalized keywords from your resume? After your first run, try `python extract_keywords.py` (in the cell at the bottom) — it suggests a tighter keyword list aligned to your background.

In [ ]:
#@title Search configuration { display-mode: "form" }

#@markdown ### Sources
greenhouse_companies = "recursionpharmaceuticals,ginkgobioworks,absci,flatironhealth,freenome,natera,komodohealth,doximity"  #@param {type:"string"}
muse_categories = "Data and Analytics,Healthcare,Project Management,Science and Engineering"  #@param {type:"string"}
muse_levels = "Senior Level,Mid Level"  #@param {type:"string"}
muse_locations = "Flexible / Remote"  #@param {type:"string"}

#@markdown ### Filters
keywords = "data,AI,digital,bioinform,computational,regulatory,product manager,business analyst"  #@param {type:"string"}
posted_within_hours = 168 #@param {type:"integer"}

#@markdown ### Pre-rank (Haiku 4.5)
prerank_enabled = True #@param {type:"boolean"}
prerank_threshold = 5 #@param {type:"slider", min:0, max:10, step:1}
prerank_max_candidates = 100 #@param {type:"slider", min:10, max:300, step:10}

#@markdown ### Deep analysis
max_jobs = 10 #@param {type:"slider", min:1, max:30, step:1}

import yaml
config = {
    "greenhouse": {"companies": [c.strip() for c in greenhouse_companies.split(",") if c.strip()]},
    "muse": {
        "categories": [c.strip() for c in muse_categories.split(",") if c.strip()],
        "levels": [l.strip() for l in muse_levels.split(",") if l.strip()],
        "locations": [l.strip() for l in muse_locations.split(",") if l.strip()],
        "max_pages": 2,
    },
    "keywords": [k.strip() for k in keywords.split(",") if k.strip()],
    "max_jobs": max_jobs,
    "posted_within_hours": posted_within_hours if posted_within_hours > 0 else None,
    "prerank": {
        "enabled": prerank_enabled,
        "threshold": prerank_threshold,
        "max_candidates": prerank_max_candidates,
    },
}
with open("config.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)
print(f"✓ Config written. Will analyze up to {max_jobs} jobs.")

## 4. Run the pipeline

This takes 1–2 minutes (scrape + pre-rank with Haiku + deep-analyze with Sonnet 4.6 + embedding similarity).

First-time downloads: O*NET skill catalog (~80MB) and the embedding model (~80MB) — adds ~10s to the first run only.

Cost: ~$0.30 in Anthropic API spend.

In [ ]:
!python pipeline.py

## 5. Browse + download results

In [ ]:
import pandas as pd
from google.colab import files
import os

if not os.path.exists("results.csv"):
    raise SystemExit("results.csv not found. Did the previous cell complete successfully?")

df = pd.read_csv("results.csv")
print(f"Top matches by qualification score:\n")
display(df[["score", "verdict", "ats_score", "domain_fit_score", "title", "company", "missing"]].head(15))

print("\nApply links (top 5):")
for _, r in df.head(5).iterrows():
    print(f"  {int(r['score']):>3}/100  [{r['verdict']}]  {r['title']} @ {r['company']}")
    print(f"          {r['url']}")

print("\n📥 Downloading results.csv and results.json...")
files.download("results.csv")
files.download("results.json")

## 6. (Optional) Tailor your resume for top matches

The model is explicitly told NOT to invent skills, inflate experience, or fabricate credentials. Tailoring is *emphasis and vocabulary*, not fiction. Cost: ~$0.05 per tailored resume.

In [ ]:
#@title Tailor settings
top_n = 3 #@param {type:"slider", min:1, max:10, step:1}
min_score = 50 #@param {type:"slider", min:0, max:100, step:5}

!python tailor.py --top {top_n} --min-score {min_score}

import os
from google.colab import files
if os.path.exists("tailored"):
    tailored_files = sorted(os.listdir("tailored"))
    print(f"\n📥 Downloading {len(tailored_files)} tailored resume(s)...")
    for fn in tailored_files:
        files.download(f"tailored/{fn}")
    print("\nTo convert to DOCX for ATS upload (do locally — pandoc isn't in Colab by default):")
    print("    pandoc tailored/foo.md -o foo.docx")

## 🎉 All done

**Source repo:** https://github.com/aa9gj/fitcast

The full CLI version offers more (`python audit.py <url>` to inspect any score's derivation, `python check_resume_format.py` to test your real PDF, `python track.py mark <url>` to track applications, etc.) — see the README.

**Re-running:** change form values in any cell and re-run from there (Runtime → Run after).

**Reset everything:** Runtime → Disconnect and delete runtime, then re-open this notebook.